# Predicting and Inferring Climate Metrics in the United States:
# A 2016 Weekly Analysis of Temperature and Precipitation

## Data Collection

This study uses weather data from the National Weather Service (NWS), a division of the National Oceanic and Atmospheric Administration (NOAA), which collects daily weather observations through Weather Forecast Offices across the United States. The data, provided by the CORGIS Dataset Project, summarizes these observations at a weekly level for cities nationwide throughout 2016.

Using this dataset, I examine seasonal patterns in temperature and precipitation, explore relationships between different weather variables, build predictive models for average weekly temperature, and perform inferential analyses to compare climate differences between regions.합니다.

## Problem Formulation

### Descriptive Question

Weather conditions in the United States tend to follow clear seasonal patterns throughout the year. In this section, I start by examining monthly averages of temperature and precipitation for 2016 in order to understand how these variables evolve over time. Summarizing the data at the monthly level allows me to focus on broad trends rather than short-term fluctuations, helping to establish an overall picture of national climate behavior that will guide the analyses that follow.

### Exploratory Question

Temperature and precipitation are often thought to be related, but this relationship does not always appear clearly in real data. In some periods, temperatures can be high while there is little rainfall, while in other cases, lower temperatures are accompanied by relatively high levels of precipitation. These observations suggest that the relationship between the two is not simple and that other factors, such as season or timing, may also be playing a role. For this reason, I focus here on looking at temperature and precipitation together to see whether any general patterns emerge and whether noticeable differences appear across different times of the year.

### Predictive Question

In this section, I aim to predict average weekly temperature using month, wind speed, and precipitation as explanatory variables. I begin with a linear regression model because it provides a simple and interpretable baseline for understanding how each predictor is linearly related to temperature. Using linear regression first allows me to assess whether these variables have meaningful predictive power and to establish a reference level of model performance. This baseline model then serves as a point of comparison for more flexible, non-linear approaches, helping to evaluate whether increased model complexity leads to meaningful improvements in prediction accuracy.

### Inferential Question

When comparing climate patterns across the United States, differences between the East and the West often seem obvious, but those impressions can be influenced by how the data are sampled. Average temperatures can shift noticeably from year to year and across seasons, so an apparent gap between California in the West and New York in the East may not necessarily reflect a consistent regional difference. To avoid putting too much weight on a single observed average, I use a resampling approach that repeatedly draws from the data. By examining how the difference in mean temperatures between the Western and Eastern regions behaves across these repeated samples, I can get a more personal and intuitive sense of whether the temperature gap is stable or if it largely comes from random variation in the data.

## Data Preparation

In [14]:
import pandas as pd
import numpy as np

weather = pd.read_csv('Weather.csv')

weather.columns = [c.split('.')[-1].strip() for c in weather.columns]

weather_tidy = weather[['Avg Temp', 'Precipitation', 'Speed', 'Month', 'State', 'Year']].dropna()

weather_tidy = weather_tidy.rename(columns={
    'Avg Temp': 'Avg_Temp',
    'Speed': 'Wind_Speed'
})

weather_tidy = weather_tidy[weather_tidy['Year'] == 2016]

def get_season(month):
    if month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    elif month in [9, 10, 11]: return 'Fall'
    else: return 'Winter'

weather_tidy['Season'] = weather_tidy['Month'].apply(get_season)

print(weather_tidy.head())

   Avg_Temp  Precipitation  Wind_Speed  Month    State  Year  Season
0        39           0.00        4.33      1  Alabama  2016  Winter
1        39           0.00        3.86      1  Alabama  2016  Winter
2        46           0.16        9.73      1  Alabama  2016  Winter
3        45           0.00        6.86      1  Alabama  2016  Winter
4        34           0.01        7.80      1   Alaska  2016  Winter


This code cleans and prepares the weather dataset for analysis by selecting key variables, renaming columns for clarity, and removing missing values. The data is then filtered to include only observations from 2016, and a new season variable is created based on the month. The resulting tidy dataset is ready for use in descriptive, exploratory, predictive, and inferential analyses.

In [15]:
import altair as alt

monthly_summary = weather_tidy.groupby('Month')[['Avg_Temp', 'Precipitation']].mean().reset_index()

temp_line = alt.Chart(monthly_summary, title="Monthly Average Temperature in 2016").mark_line(point=True).encode(
    x=alt.X("Month:O").title("Month"),
    y=alt.Y("Avg_Temp:Q").scale(zero=False).title("Average Temperature (F)")
).properties(width=500, height=300)

precip_bar = alt.Chart(monthly_summary, title="Monthly Average Precipitation in 2016").mark_bar().encode(
    x=alt.X("Month:O").title("Month"),
    y=alt.Y("Precipitation:Q").title("Precipitation (Inches)")
).properties(width=500, height=300)

combined_plot = alt.vconcat(temp_line, precip_bar).configure_axis(titleFontSize=12)
combined_plot.display()

alt.VConcatChart(...)

The visualizations show clear seasonal patterns in both temperature and precipitation. Average monthly temperature rises steadily from the beginning of the year into the summer months, peaking around July and August, and then gradually decreases toward the end of the year. This trend matches what would generally be expected for temperature changes across the United States. Precipitation appears more variable throughout the year, with some months showing relatively similar levels and others standing out with higher average rainfall, particularly in the late summer. Overall, these plots help illustrate the strong seasonal cycle in temperature and the more uneven pattern observed in precipitation.

In [20]:
import altair as alt
import warnings
warnings.filterwarnings('ignore')

alt.data_transformers.disable_max_rows()

exploratory_scatter = alt.Chart(weather_tidy, title="Relationship between Temperature and Precipitation by Season").mark_circle(size=60, opacity=0.4).encode(
    x=alt.X("Avg_Temp:Q").title("Average Temperature (F)"),
    y=alt.Y("Precipitation:Q").title("Precipitation (Inches)"),
    color=alt.Color("Season:N").scale(scheme="dark2").title("Season")
).properties(
    width=600,
    height=400
).interactive()

exploratory_scatter.display()

alt.Chart(...)

The scatter plot suggests that temperature and precipitation do not follow a simple relationship. Precipitation varies widely across both low and high temperature ranges, with no clear trend visible overall. Seasonal grouping, however, shows that warmer months tend to cluster at higher temperatures, while cooler months appear more tightly grouped at lower temperatures. This visualization highlights the variability in precipitation and the role of season in shaping the observed patterns.

In [29]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

train_df, test_df = train_test_split(weather_tidy, train_size=0.75, random_state=1)

X_train = train_df[["Month", "Wind_Speed", "Precipitation"]]
y_train = train_df["Avg_Temp"]
X_test = test_df[["Month", "Wind_Speed", "Precipitation"]]
y_test = test_df["Avg_Temp"]

lm_model = LinearRegression()
lm_model.fit(X_train, y_train)
lm_r2 = lm_model.score(X_test, y_test)
lm_rmspe = mean_squared_error(y_test, lm_model.predict(X_test))**(1/2)

knn_model = KNeighborsRegressor(n_neighbors=55)
knn_model.fit(X_train, y_train)
knn_rmspe = mean_squared_error(y_test, knn_model.predict(X_test))**(1/2)

print(f"RMSPE")
print(f"Linear Regression RMSPE: {lm_rmspe:.4f}")
print(f"K-NN Regression RMSPE: {knn_rmspe:.4f}")
print(f"Linear Regression R-squared: {lm_r2:.4f}")

RMSPE
Linear Regression RMSPE: 17.7472
K-NN Regression RMSPE: 12.0588
Linear Regression R-squared: 0.0805


In [28]:
import pandas as pd
import numpy as np
import altair as alt

x_grid = pd.DataFrame({"Month": [weather_tidy["Month"].min(), weather_tidy["Month"].max()]})

x_grid["Wind_Speed"] = weather_tidy["Wind_Speed"].mean()
x_grid["Precipitation"] = weather_tidy["Precipitation"].mean()

x_grid["predicted"] = lm_model.predict(x_grid[["Month", "Wind_Speed", "Precipitation"]])

points = alt.Chart(weather_tidy).mark_circle(opacity=0.3, color="gray").encode(
    x=alt.X("Month:Q").title("Month"),
    y=alt.Y("Avg_Temp:Q").title("Average Temperature (F)")
)

reg_line = alt.Chart(x_grid).mark_line(color="#ff7f0e", size=3).encode(
    x="Month:Q",
    y="predicted:Q"
)

final_plot = (points + reg_line).properties(
    title="Linear Regression: Temperature Trend by Month",
    width=500,
    height=300
).configure_axis(titleFontSize=12)

final_plot.display()

alt.LayerChart(...)

Looking at the RMSPE results, the linear regression model has an RMSPE of 17.7472, which means that, on average, the predicted temperature is off from the actual temperature by about 18°F. In contrast, the K-NN regression model has a lower RMSPE of 12.0588, indicating that it makes more accurate predictions than the linear regression model. In addition, the linear regression model’s R-squared value is only 0.0805, showing that the variables Month, Wind Speed, and Precipitation explain only about 8% of the variation in average temperature. Overall, these numbers suggest that using this set of variables with a linear regression model is not sufficient for accurately predicting temperature.

This conclusion becomes clearer when looking at the graph. While there is a clear seasonal pattern where average temperature generally increases as the months progress from winter to summer, the actual temperature values within each month are spread out quite widely. In other words, even within the same month, temperatures can vary a lot. The regression line captures only the overall average trend, but most data points are scattered far from the line rather than clustering around it. This makes it clear that Month alone cannot explain individual temperature observations, and that other factors—such as location or day-to-day weather conditions—play a large role. This visual pattern helps explain why the R-squared value is low and why the prediction error remains relatively large.

In [40]:
import pandas as pd
import numpy as np
import altair as alt

top_5_states = weather_tidy['State'].value_counts().nlargest(5).index.tolist()
print(f"Top 5 States by observation count: {top_5_states}")

states_to_compare = ['Florida', 'California']

def run_bootstrap_diff(data, state_a, state_b, season, n_replicates=20000):
    season_data = data[data['Season'] == season]
    a_temps = season_data[season_data['State'] == state_a]['Avg_Temp']
    b_temps = season_data[season_data['State'] == state_b]['Avg_Temp']
    
    observed_diff = a_temps.mean() - b_temps.mean()
    
    np.random.seed(42)
    boot_diffs = []
    for _ in range(n_replicates):
        a_boot = a_temps.sample(frac=1, replace=True)
        b_boot = b_temps.sample(frac=1, replace=True)
        boot_diffs.append(a_boot.mean() - b_boot.mean())
    
    boot_results = pd.DataFrame({"diff_means": boot_diffs, "Season": season})
    ci = boot_results["diff_means"].quantile([0.025, 0.975])
    return boot_results, observed_diff, ci

results = []
for season in ['Summer', 'Winter']:
    boot_df, obs, ci = run_bootstrap_diff(weather_tidy, states_to_compare[0], states_to_compare[1], season)
    results.append((boot_df, obs, ci, season))

summer_res = results[0]
summer_chart = alt.Chart(summer_res[0]).mark_bar(opacity=0.6, color='orange').encode(
    x=alt.X("diff_means:Q").bin(maxbins=40).title("Summer Mean Diff (FL - CA, F)"),
    y=alt.Y("count()").title("Frequency")
).properties(title="Summer Bootstrap Distribution", width=400, height=300)

s_obs_line = alt.Chart(pd.DataFrame({'x': [summer_res[1]]})).mark_rule(color='blue', size=2).encode(x='x')
s_ci_lines = alt.Chart(pd.DataFrame({'x': [summer_res[2][0.025], summer_res[2][0.975]]})).mark_rule(color='red', strokeDash=[5,5], size=2).encode(x='x')

(summer_chart + s_obs_line + s_ci_lines).display()

winter_res = results[1]
winter_chart = alt.Chart(winter_res[0]).mark_bar(opacity=0.6, color='skyblue').encode(
    x=alt.X("diff_means:Q").bin(maxbins=40).title("Winter Mean Diff (FL - CA, F)"),
    y=alt.Y("count()").title("Frequency")
).properties(title="Winter Bootstrap Distribution", width=400, height=300)

w_obs_line = alt.Chart(pd.DataFrame({'x': [winter_res[1]]})).mark_rule(color='blue', size=2).encode(x='x')
w_ci_lines = alt.Chart(pd.DataFrame({'x': [winter_res[2][0.025], winter_res[2][0.975]]})).mark_rule(color='red', strokeDash=[5,5], size=2).encode(x='x')

(winter_chart + w_obs_line + w_ci_lines).display()

for _, obs, ci, season in results:
    print(f"\n[{season}] {states_to_compare[0]} - {states_to_compare[1]}")
    print(f"Observed Difference: {obs:.4f} F")
    print(f"95% Confidence Interval: [{ci[0.025]:.4f}, {ci[0.975]:.4f}]")

Top 5 States by observation count: ['Alaska', 'Texas', 'California', 'Florida', 'Montana']


alt.LayerChart(...)

alt.LayerChart(...)


[Summer] Florida - California
Observed Difference: 10.8846 F
95% Confidence Interval: [9.7594, 11.9933]

[Winter] Florida - California
Observed Difference: 11.2558 F
95% Confidence Interval: [9.7300, 12.8083]


Looking at the bootstrap results, the difference in average temperature between Florida and California stays consistently positive across seasons. In the summer, the estimated mean difference is around 10.9°F, and the bootstrap distribution is clearly centered near this value. The 95% confidence interval, which runs roughly from 9.8°F to 12.0°F, is fairly narrow, making it seem unlikely that this difference is just due to random chance. The fact that the distribution is fairly symmetric and has a single clear peak also makes the estimate feel quite stable.

The winter results show a similar pattern, but with a slightly larger gap. The estimated winter mean difference is about 11.3°F, and again the 95% confidence interval does not include zero. To me, this strongly suggests that Florida remains warmer than California even in the winter, and that this pattern is consistent rather than seasonal noise. Seeing both summer and winter intervals clearly separated from zero makes the difference feel meaningful regardless of season.

Overall, this bootstrap graph feels more convincing than just comparing raw averages. The way the values cluster within a limited range and the clear separation of the confidence intervals makes it seem like the temperature difference between the West and the East is pretty steady. Personally, I find this visualization persuasive, since it suggests that the gap is driven by underlying regional climate differences rather than random fluctuations in the data.

## Problem Reformulation

